[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Host_Prog/Intro_OS/Intro_OS.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Intro to Operating Systems

Every benchmark you've run in this curriculum — the NumPy timings in [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb), the GPU transfers in [Intro to GPU Systems](../../Intro_GPU/Intro_GPU.ipynb) — happened *on top of* an operating system quietly scheduling, paging, and buffering underneath you. This workshop makes that layer visible: processes, memory, concurrency, and the shell.

## 0. Introduction

An OS does three jobs:

1. **Abstraction** — files instead of disk blocks, processes instead of CPU time slices.
2. **Multiplexing** — hundreds of programs sharing a few cores and one memory.
3. **Protection** — your buggy pointer arithmetic ([Intro to C](../../Intro_Programming/Intro_C.ipynb) §5) crashes *your* process, not the machine.

We poke at all three with live code.

## 1. Pre-requisites

- [Intro to Python](../../Intro_Programming/Intro_Python/Intro_Python.ipynb); [Intro to C](../../Intro_Programming/Intro_C.ipynb) for the memory sections.
- **A Unix-like environment**: Linux, macOS, WSL on Windows, or Google Colab (which is Linux underneath — every cell here runs there). Windows-native Python will fail on the `/proc` cells.

---
### 🕐 Session 1 of 4 — *Processes & the Kernel* (~35 min)
**Goal:** see what a process is, watch syscalls happen, and understand user vs kernel mode.
**Feeds into:** Session 2 (memory & scheduling).

---

## 2. Processes

💡 **Intuition.** A *program* is a file; a *process* is that program **caught in the act**: code plus its memory, open files, and a kernel-side identity (the PID). The kernel is the only code with full hardware access; your process must *ask* for anything beyond arithmetic — every file read, print, and allocation is a **syscall**, a controlled doorbell into the kernel.

In [ ]:
import os, sys

print("my PID:", os.getpid())
print("my parent's PID:", os.getppid(), "(the kernel/jupyter that launched me)")
print("running as user:", os.getuid())
print("current working dir:", os.getcwd())

### 2.1. The Process Tree

Every process is spawned by another — back to PID 1. Your notebook kernel is a child of the Jupyter server, which is a child of your shell…

In [ ]:
# walk our own ancestry via /proc

# YOUR CODE HERE


### 2.2. Watching Syscalls

`strace` prints every doorbell a process rings. Even a do-nothing program makes dozens of syscalls just to start up.

In [ ]:

# YOUR CODE HERE


Read the table: `read`, `write`, `mmap`, `openat` — *file access and memory mapping dominate even a hello-world*. This is why 'minimize syscalls' is a real optimization strategy in real-time signal processing.

---
### 🕐 Session 2 of 4 — *Memory & Scheduling* (~35 min)
**Goal:** understand virtual memory and the scheduler — and why your benchmark numbers wobble.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (concurrency).

---

## 3. Virtual Memory

💡 **Intuition.** Every process believes it owns the *entire* address space — a private, contiguous billboard of bytes starting at (almost) zero. It's a lie the hardware maintains: the **page table** maps each 4 KB *virtual* page to wherever the kernel actually parked it in RAM (or on disk!). The lie is what makes [C pointers](../../Intro_Programming/Intro_C.ipynb) safe to hand out: your address 0x5000 and my address 0x5000 are different physical bytes.

In [ ]:
# Our own memory map: every region the kernel granted this process

# YOUR CODE HERE


**What just happened.** `/proc/self/maps` listed 154 separate memory regions for a process that hasn't done anything yet — the Python interpreter binary, `libpython`, shared libraries, the stack, and (last line) the `[vsyscall]` page. Each line is one entry in the page table's bookkeeping: a virtual address range, its permissions (`r`/`w`/`x`), and where it's backed (a file on disk, or nothing for anonymous memory like the stack). The takeaway isn't the exact count — it's that *before you've allocated a single Python object*, the kernel has already built a substantial map just to load the interpreter, and every one of those regions is virtual: real physical RAM is only committed when a page inside it is actually touched, which the next cell demonstrates directly.

In [ ]:
# Allocation is lazy: address space is cheap, physical pages are charged on TOUCH

# YOUR CODE HERE


`np.empty(200 MB)` cost almost nothing — the kernel handed out *promises*, not pages. Touching the pages forced it to deliver. Moral for benchmarking: **first-touch cost lands on your first iteration**, which is (one reason) why the GPU notebook told you to run cells twice.

## 4. The Scheduler

💡 **Intuition.** More runnable threads than cores ⇒ someone waits. The scheduler slices CPU time and swaps processes on and off cores; each swap (*context switch*) trashes caches and costs microseconds. Your timing jitter is mostly *other people's processes*.

In [ ]:
# Measure scheduler jitter: ask to sleep 1 ms, see what we actually get

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *Concurrency in Practice* (~40 min)
**Goal:** spawn threads and processes; cause (and fix) a race condition; build a producer/consumer pipeline.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (the shell).

---

## 5. Threads, Races, Locks

💡 **Intuition.** A **race condition** is two workers doing read-modify-write on the same data with no coordination: both read 5, both write 6, one increment vanishes. The bug is *probabilistic* — it disappears when you look closely (add a print → timing changes → race hides). That's why the fix is discipline (locks), not debugging.

In [ ]:
        # the gap between read and write is where disaster lives;
        # we widen it so the race shows up reliably in a demo

# YOUR CODE HERE


In [ ]:

# YOUR CODE HERE


**What just happened.** The unsafe version above lost 147,000 of 200,000 increments (`53,000` survived) — four threads each doing `v = counter; ...; counter = v + 1` with no coordination means two threads can both read the same value of `counter` before either writes back, so one thread's increment silently overwrites the other's. Wrapping the read-modify-write in `with lock:` forces the whole triplet to run as one uninterruptible unit per thread, and the count comes back exact: `200,000`. Nothing about the *arithmetic* changed — only who's allowed to interleave when. This is also why the loss rate isn't deterministic or predictable from first principles: it depends on exactly how the scheduler happens to interleave the four threads, which is precisely the "probabilistic bug" property flagged above.

### 5.1. Threads vs Processes in Python

CPython's **GIL** lets only one thread run Python bytecode at a time — threads buy you *concurrency* (overlapping waits) but not *parallelism* for pure-Python math. For CPU-bound work, use **processes** (separate memory, true parallelism) — or NumPy, which releases the GIL inside its C loops. This is why the [GPU workshop's](../../Intro_GPU/Intro_GPU.ipynb) vectorization advice works even from single-threaded Python.

In [ ]:
    # pure-Python CPU work: sum of squares

# YOUR CODE HERE


### 5.2. Producer/Consumer

The queue is the honest way for concurrent workers to talk: no shared mutable state, no races — the pattern behind every data-acquisition pipeline (sensor thread produces, processing thread consumes).

In [ ]:

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *The Shell & Automation* (~35 min)
**Goal:** compose tools with pipes; control processes; package an experiment so it reruns cleanly.
**Builds on:** Session 1.

---

## 6. The Shell

💡 **Intuition.** The pipe `|` is the shell's superpower: each program reads a stream and writes a stream, and the kernel plumbs them together — tiny tools composing into pipelines, with the OS scheduling all stages *concurrently*. It's the producer/consumer queue from Session 3, built into the operating system.

In [ ]:
# Pipeline: of the 5 most memory-hungry processes, show name + resident MB

# YOUR CODE HERE


In [ ]:
# Redirection + exit codes: the glue of automation

# YOUR CODE HERE


### 6.1. Environment & Reproducibility

Environment variables are the OS-level configuration channel — the reason `conda activate` works and half of all "works on my machine" bugs exist.

In [ ]:

# YOUR CODE HERE


**What just happened.** `PATH` resolved to 13 directories the shell searches in order to find `python3` — the first match wins, which is the entire mechanism behind "wrong Python got picked up" bugs when a `conda`/`venv` environment isn't activated (or is shadowed by something earlier in `PATH`). The second line shows environment variables crossing the process boundary from Session 1: `MY_EXPERIMENT_SEED=42` set in front of the command is inherited by the *child* process, which reads it back out of its own `os.environ`. Environment variables are a one-way, copy-on-fork channel — the parent shell's variable and the child's `os.environ['MY_EXPERIMENT_SEED']` are independent copies of the same string, not a shared reference; the child can't write back to the parent's environment. This is the OS-level mechanism underneath every `--seed`/`--lr` flag and every `conda activate` in the reproducibility checklist below.

### 6.2. The Reproducible-Experiment Checklist

Package every experiment so future-you can rerun it:

1. **Pin the environment** — `requirements.txt` / `environment.yml`, committed.
2. **Take config from argv/env, not edits** — `python train.py --lr 0.01 --seed 42` ([Intro to C §8](../../Intro_Programming/Intro_C.ipynb) taught the same interface).
3. **Log to files, exit nonzero on failure** — so shell scripts and cron can react.
4. **Write results to a database, not scattered CSVs** — the [Databases workshop](../Intro_Databases/Intro_Databases.ipynb) built exactly this logger.


## 7. Conclusion

Processes are running programs with kernel-side identity; memory is a per-process illusion maintained by page tables; schedulers cause your timing jitter; races die by lock or by queue; and the shell composes it all. You now know what's *underneath* every other workshop.

---
## Where next

- [Intro to Databases](../Intro_Databases/Intro_Databases.ipynb) — files + locks + transactions, industrialized.
- [Intro to GPU Systems](../../Intro_GPU/README.md) — a second processor with its own memory hierarchy to schedule.
- [Intro to C](../../Intro_Programming/Intro_C.ipynb) — the language the kernel itself is written in.